In [2]:
from osgeo import gdal

# Open the raster dataset
dataset = gdal.Open('/home/jovyan/Downloads/arcos.tiff')

# Check if file opened successfully
if dataset is not None:
    # Get basic dimensions
    cols = dataset.RasterXSize
    rows = dataset.RasterYSize
    bands = dataset.RasterCount
    
    # Get geographic transformation (top-left x, pixel width, rotation, top-left y, rotation, pixel height)
    geotransform = dataset.GetGeoTransform()
    
    print(f"Size: {cols} x {rows} with {bands} bands")
    print(f"Origin (X, Y): {geotransform[0]}, {geotransform[3]}")
    print(dataset)

# Close the dataset
dataset = None

Size: 2326 x 1748 with 3 bands
Origin (X, Y): 0.0, 0.0
<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x702478c98db0> >


In [3]:
def converter_tiff_para_zstd(caminho_entrada, caminho_saida):
    # Abre a imagem original
    
    caminho_entrada = ('/home/jovyan/Downloads/arcos.tiff')
    caminho_saida = ('/home/jovyan/Documents/')
    dataset = gdal.Open(caminho_entrada)
    if not dataset:
        print(f"Erro ao abrir o arquivo {caminho_entrada}")
        return False

    # Define as opções de criação para ZSTD
    opcoes_criacao = [
        "COMPRESS=ZSTD",
        "ZSTD_LEVEL=9",  # Nível de compressão (1 a 22)
        "TILED=YES"      # Organiza o raster em blocos (tiles)
    ]

    print(f"Convertendo {caminho_entrada} para ZSTD...")
   
    # Converte e salva o arquivo
    driver = gdal.GetDriverByName("GTiff")
    dataset_saida = driver.CreateCopy(caminho_saida, dataset, options = opcoes_criacao)

    if dataset_saida:
        print(f"Sucesso! Arquivo salvo em: {caminho_saida}")
        # Libera a memória
        dataset = None
        dataset_saida = None
        return True
    else:
        print(f"Erro ao converter o arquivo para {caminho_saida}")
        return False

In [4]:
#1. Reading Raster Metadata and Data
#Este fluxo de trabalho mostra como abrir um GeoTIFF, buscar suas dimensões de coordenadas, ler a projeção e carregar a banda raster em uma matriz NumPy

In [5]:
from osgeo import gdal

# Tell GDAL to throw Python exceptions on errors instead of silent failures
gdal.UseExceptions()

# Open the raster file using a context manager (closes file automatically)
with gdal.Open('/home/jovyan/Downloads/arcos.tiff') as dataset:
#with gdal.Open("input.tif") as dataset:
    # Get basic dimensions
    width = dataset.RasterXSize
    height = dataset.RasterYSize
    bands = dataset.RasterCount
    print(f"Dimensions: {width}x{height} | Bands: {bands}")
    
    # Get spatial metadata
    geotransform = dataset.GetGeoTransform()
    projection = dataset.GetProjection()
    print(f"Origin X: {geotransform[0]}, Origin Y: {geotransform[3]}")
    print(f"Pixel Size: {geotransform[1]}x{geotransform[5]}")
    
    # Fetch the first band (GDAL bands use 1-based indexing)
    band = dataset.GetRasterBand(1)
    
    # Read the data natively into a memory array
    data_array = band.ReadAsArray()
    print(f"Array structure: {data_array.shape}")

Dimensions: 2326x1748 | Bands: 3
Origin X: 0.0, Origin Y: 0.0
Pixel Size: 1.0x1.0
Array structure: (1748, 2326)


In [6]:
#2. Running GDAL Command Line Tools inside Python
#O módulo gdal inclui invólucros de alto nível para suas principais ferramentas de linha de comando (como gdalwarp ou gdal_translate).
#Estes são ideais para reprojetar, cortar ou comprimir rasters sem escrever loops manuais

In [9]:
#3. Creating a New Raster from Scratch
#Se você calcular novas matrizes de dados, use um driver GDAL para inicializar e salvar um conjunto de dados espacial limpo.

In [10]:
import numpy as np
from osgeo import gdal

gdal.UseExceptions()

# Create dummy matrix data (e.g., 512x512 pixels)
new_data = np.random.randint(0, 255, size=(512, 512), dtype=np.uint8)

# Initialize the GeoTIFF driver
driver = gdal.GetDriverByName("GTiff")

# Create blank file: path, width, height, band count, data type
output_ds = driver.Create("output_generated.tif", 512, 512, 1, gdal.GDT_Byte)

# Define spatial orientation parameters
# [Top-left X, West-East resolution, X rotation, Top-left Y, Y rotation, North-South resolution]
output_ds.SetGeoTransform([440720.0, 30.0, 0.0, 3751320.0, 0.0, -30.0])

# Write data array to the first band
out_band = output_ds.GetRasterBand(1)
out_band.WriteArray(new_data)

# Explicitly clear variables to flush data and close files cleanly
out_band = None
output_ds = None
print("Raster saved successfully.")

Raster saved successfully.


In [11]:
import os
import rasterio


def converter_para_zstd(caminho_entrada, caminho_saida, zstd_level=3):
    """Lê uma imagem .tif e a salva novamente utilizando compressão ZSTD.

    :param caminho_entrada: Caminho para o arquivo TIFF original.
    :param caminho_saida: Caminho onde o novo arquivo convertido será salvo.
    :param zstd_level: Nível de compressão ZSTD (1 a 22). O padrão é 3.
    """
    if not os.path.exists(caminho_entrada):
        print(f"Erro: O arquivo de entrada '{caminho_entrada}' não existe.")
        return

    print(f"Lendo '{caminho_entrada}'...")

    # 1. Abrir o arquivo original para ler os dados e o perfil geoespacial
    with rasterio.open(caminho_entrada) as src:
        # Copia o perfil (metadados) do arquivo original
        perfil = src.profile.copy()

        # Lê todas as bandas da imagem de uma vez
        dados_imagem = src.read()

    # 2. Atualizar o perfil com as configurações de compressão ZSTD
    perfil.update(
        compress="zstd",
        zstd_level=zstd_level,
        # O predictor=2 (horizontal differencing) ajuda muito a diminuir o tamanho
        # de arquivos com dados contínuos (ex: modelos de elevação ou imagens de satélite)
        predictor=2,
    )

    print(f"Salvando com compressão ZSTD (Nível {zstd_level}) em: '{caminho_saida}'...")

    # 3. Gravar o novo arquivo com o perfil atualizado
    with rasterio.open(caminho_saida, "w", **perfil) as dst:
        dst.write(dados_imagem)

    print("Conversão concluída com sucesso!")


# --- Exemplo de Uso ---
if __name__ == "__main__":
    arquivo_original = "/home/jovyan/Downloads/arcos.tiff"
    arquivo_compactado = "arcos_zstd.tiff"

    converter_para_zstd(arquivo_original, arquivo_compactado, zstd_level=3)

Lendo '/home/jovyan/Downloads/arcos.tiff'...
Salvando com compressão ZSTD (Nível 3) em: 'arcos_zstd.tiff'...


/home/jovyan/.local/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/home/jovyan/.local/lib/python3.11/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


Conversão concluída com sucesso!
